# Ecommerce product details automation
This creates compelling product listings for e-commerce company selling thousands of products. Each product gets:
- A catchy title
- A detailed description
- Key features highlighted
- SEO-friendly content

In [ ]:
!uv pip install datasets openai python-dotenv

Using Python 3.12.15 environment at: /opt/miniconda3/envs/ironhack
Checked 1 package in 86ms


### Step 1: Setting Up OpenAI API Access
Added .env and .gitignore 

### Step 2: Preparing the Dataset


In [2]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    ds = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")
    print(f"✓ Loaded {len(ds)} products")
    products_df = pd.DataFrame(ds)
    print(f"Dataset columns: {products_df.columns.tolist()}")
except Exception as e:
    print(f"✗ Error loading dataset: {e}")


# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")
print(products_df[0:1])

/opt/miniconda3/envs/ironhack/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100
      id gender masterCategory subCategory articleType baseColour season  \
0  15970    Men        Apparel     Topwear      Shirts  Navy Blue   Fall   

     year   usage                productDisplayName  \
0  2011.0  Casual  Turtle Check Men Navy Blue Shirt   

                                               image  
0  <PIL.JpegImagePlugin.JpegImageFile image mode=...  


### Step 3: Encoding Images for API

In [ ]:
import base64
from io import BytesIO

def encode_image_to_base64(image, format="JPEG"):
    """Encode a PIL image to a base64 string."""
    buffer = BytesIO()
    # Some images are grayscale ("L") or have alpha, so normalise to RGB for JPEG
    image.convert("RGB").save(buffer, format=format)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# Encode every product image and store it alongside the metadata
products_df["image_b64"] = products_df["image"].apply(encode_image_to_base64)

print(f"✓ Encoded {products_df['image_b64'].notna().sum()} / {len(products_df)} images")
print(f"Encoded image length (product 0): {len(products_df.iloc[0]['image_b64'])} characters")
print(f"Encoded prefix: {products_df.iloc[0]['image_b64'][:40]}...")

✓ Encoded 100 / 100 images
Encoded image length (product 0): 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


### Step 4: Creating the Product Listing Prompt

In [7]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Create a well rounded product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}

For the product description, be specific about what you see in the image and highlight key features and benefits. 
Mention colors, materials, design elements, and any distinctive features. 
Put yourself in the shoes of the customer - what would make them want to buy this product. 
Use subtle persuasive language that highlights the benefits of the product and what makes it special. 
Avoid generic phrases like "high quality" or "best product". Instead, focus on unique selling points and specific details that make this product stand out."""
    
    return prompt

# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE

You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Create a well rounded product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** ...


### Step 5: Calling the ChatGPT API with Vision

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from the .env file in this folder
client = OpenAI()  # picks up OPENAI_API_KEY from the environment automatically

# The dataset has no price, so assign a placeholder price by masterCategory
PLACEHOLDER_PRICES = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99,
}
DEFAULT_PRICE = 19.99  # anything else

products_df["price"] = products_df["masterCategory"].map(PLACEHOLDER_PRICES).fillna(DEFAULT_PRICE)
print(products_df.groupby("masterCategory")["price"].first())

def generate_listing(row, model="gpt-4o-mini"):
    prompt = create_product_listing_prompt(
        product_name=row["productDisplayName"],
        price=row["price"],
        category=f"{row['masterCategory']} / {row['articleType']}",
        additional_info=f"Colour: {row['baseColour']}, Usage: {row['usage']}",
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{row['image_b64']}"}},
            ],
        }],
        response_format={"type": "json_object"},  # forces valid JSON output
        temperature=0.7,
    )
    return response.choices[0].message.content

def parse_listing(raw_text):
    # Fallback for models that wrap JSON in ```json fences (Troubleshooting Issue 4)
    cleaned = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)

raw = generate_listing(products_df.iloc[0])
listing = parse_listing(raw)
print(json.dumps(listing, indent=2))

### Step 6: Processing Multiple Products

In [ ]:
import time

N_PRODUCTS = 5  # start small to save credits
results = []

for i, row in products_df.head(N_PRODUCTS).iterrows():
    entry = {"id": int(row["id"]), "name": row["productDisplayName"], "price": float(row["price"])}
    try:
        raw = generate_listing(row)
        entry["listing"] = parse_listing(raw)
        entry["status"] = "ok"
        print(f"✓ [{i + 1}/{N_PRODUCTS}] {entry['name']}")
    except Exception as e:
        # One failure shouldn't stop the loop; record it and move on
        entry["status"] = "error"
        entry["error"] = f"{type(e).__name__}: {e}"
        print(f"✗ [{i + 1}/{N_PRODUCTS}] {entry['name']} -> {entry['error']}")
    results.append(entry)
    time.sleep(1)  # avoid rate-limit errors

with open("generated_listings.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

ok = sum(r["status"] == "ok" for r in results)
print(f"\nDone: {ok}/{len(results)} succeeded. Saved to generated_listings.json")

### Step 7: Letting the Model Estimate the Price
Instead of a placeholder price, we leave the price out of the prompt and ask the model to estimate one from the image and product details.

In [ ]:
def create_listing_prompt_with_price_estimate(product_name, category, additional_info=None):
    """Same listing prompt as Step 4, but no price is given: the model estimates it."""
    return f"""
You are an expert e-commerce copywriter and pricing analyst. Analyze the product image and create a compelling product listing, including a realistic retail price.

Product Information:
- Name: {product_name}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Create a well rounded product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
5. **Estimated Price** (a realistic retail price in USD, based on what you see in the image, the brand and the product type)
6. **Price Rationale** (one sentence explaining the estimate)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ...",
    "estimated_price": 0.00,
    "price_rationale": "One sentence explaining the estimate"
}}

"estimated_price" must be a number (no currency symbol). Suggest a price you think is realistic for this product, based on what you see in the image, the brand and the product type.

For the product description, be specific about what you see in the image and highlight key features and benefits.
Mention colors, materials, design elements, and any distinctive features.
Put yourself in the shoes of the customer - what would make them want to buy this product.
Use subtle persuasive language that highlights the benefits of the product and what makes it special.
Avoid generic phrases like "high quality" or "best product". Instead, focus on unique selling points and specific details that make this product stand out.
Do not mention the price in the description."""


def generate_listing_with_price_estimate(row, model="gpt-4o-mini"):
    prompt = create_listing_prompt_with_price_estimate(
        product_name=row["productDisplayName"],
        category=f"{row['masterCategory']} / {row['articleType']}",
        additional_info=f"Colour: {row['baseColour']}, Usage: {row['usage']}",
    )
    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{row['image_b64']}"}},
            ],
        }],
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    return response.choices[0].message.content


# Run on the same products as Step 6 so the two prices can be compared
N_PRODUCTS = 5
ai_results = []

for i, row in products_df.head(N_PRODUCTS).iterrows():
    entry = {
        "id": int(row["id"]),
        "name": row["productDisplayName"],
        "placeholder_price": float(row["price"]),
    }
    try:
        listing = parse_listing(generate_listing_with_price_estimate(row))
        entry["listing"] = listing
        entry["ai_price"] = float(listing["estimated_price"])  # fails loudly if missing/non-numeric
        entry["status"] = "ok"
        print(f"✓ [{i + 1}/{N_PRODUCTS}] {entry['name']}: "
              f"placeholder ${entry['placeholder_price']:.2f} vs AI ${entry['ai_price']:.2f}")
    except Exception as e:
        entry["status"] = "error"
        entry["error"] = f"{type(e).__name__}: {e}"
        print(f"✗ [{i + 1}/{N_PRODUCTS}] {entry['name']} -> {entry['error']}")
    ai_results.append(entry)
    time.sleep(1)  # avoid rate-limit errors

with open("generated_listings_ai_price.json", "w") as f:
    json.dump(ai_results, f, indent=2, ensure_ascii=False)

ok = sum(r["status"] == "ok" for r in ai_results)
print(f"\nDone: {ok}/{len(ai_results)} succeeded. Saved to generated_listings_ai_price.json")